# 05 — Churn Prediction Model (LightGBM)

**Inputs:** `outputs/train.parquet`, `outputs/val.parquet`, `outputs/test.parquet`  
**Outputs:** `outputs/churn_model.pkl`, `outputs/test_predictions.parquet`

| Section | What |
|---|---|
| 1 — Baseline | LightGBM `class_weight='balanced'`, val metrics |
| 2 — Imbalance | Compare balanced-weight vs SMOTE; pick winner |
| 3 — Tuning | 24-combo grid search on val (no CV — temporal integrity) |
| 4 — Final | Refit winner on train+val, evaluate on test, save |

**Features (7):** recency · frequency · monetary · aov · purchase_velocity · days_as_customer · unique_products  
**Target:** churn_label (1 = churned, 0 = retained, ~23% positive rate)

In [1]:
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from itertools import product
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
)

In [2]:
TRAIN_PATH = Path("../outputs/train.parquet")
VAL_PATH   = Path("../outputs/val.parquet")
TEST_PATH  = Path("../outputs/test.parquet")
MODEL_PATH = Path("../outputs/churn_model.pkl")
PREDS_PATH = Path("../outputs/test_predictions.parquet")

FEATURE_COLS = [
    "recency", "frequency", "monetary", "aov",
    "purchase_velocity", "days_as_customer", "unique_products",
]
TARGET_COL = "churn_label"

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    assert p.exists(), f"Run notebook 04 first — {p} not found"

train = pd.read_parquet(TRAIN_PATH)
val   = pd.read_parquet(VAL_PATH)
test  = pd.read_parquet(TEST_PATH)

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]

print(f"train : {X_train.shape}   churn = {y_train.mean()*100:.1f}%")
print(f"val   : {X_val.shape}   churn = {y_val.mean()*100:.1f}%")
print(f"test  : {X_test.shape}   churn = {y_test.mean()*100:.1f}%")

train : (2162, 7)   churn = 21.9%
val   : (380, 7)   churn = 28.7%
test  : (293, 7)   churn = 27.6%


In [3]:
def eval_metrics(model, X, y, split=""):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    return {
        "split":     split,
        "accuracy":  round(accuracy_score(y, y_pred),                    4),
        "precision": round(precision_score(y, y_pred, zero_division=0),  4),
        "recall":    round(recall_score(y, y_pred),                      4),
        "f1":        round(f1_score(y, y_pred),                          4),
        "roc_auc":   round(roc_auc_score(y, y_prob),                     4),
    }

---
## Section 1 — Baseline Model

LightGBM with `class_weight='balanced'` — the library automatically scales minority class
weights inversely proportional to its frequency. No preprocessing required.

In [4]:
baseline = LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1)
baseline.fit(X_train, y_train)

baseline_metrics = eval_metrics(baseline, X_val, y_val, split="val (baseline)")
print("Baseline — val set metrics:")
pd.DataFrame([baseline_metrics]).set_index("split")

Baseline — val set metrics:


,accuracy,precision,recall,f1,roc_auc
split,,,,,
val (baseline),0.9895,0.973,0.9908,0.9818,0.9999


---
## Section 2 — Handle Class Imbalance

Two approaches compared on the **val set**:

- **(a) `class_weight='balanced'`** — already computed above as the baseline
- **(b) SMOTE + no class_weight** — Synthetic Minority Over-sampling Technique generates new
  synthetic churned-customer samples by interpolating between existing minority examples,
  balancing the training set before fitting

Primary metric for the winner decision: **ROC-AUC** (threshold-independent).

In [5]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"Training set before SMOTE : {X_train.shape}   churn = {y_train.mean()*100:.1f}%")
print(f"Training set after  SMOTE : {X_train_sm.shape}  churn = {y_train_sm.mean()*100:.1f}%")

smote_model = LGBMClassifier(random_state=42, verbose=-1)
smote_model.fit(X_train_sm, y_train_sm)
smote_metrics = eval_metrics(smote_model, X_val, y_val, split="val (SMOTE)")

Training set before SMOTE : (2162, 7)   churn = 21.9%
Training set after  SMOTE : (3376, 7)  churn = 50.0%


In [6]:
comparison = pd.DataFrame([
    {"approach": "(a) class_weight='balanced'",
     **{k: v for k, v in baseline_metrics.items() if k != "split"}},
    {"approach": "(b) SMOTE + no class_weight",
     **{k: v for k, v in smote_metrics.items()   if k != "split"}},
]).set_index("approach")

print("Imbalance approach comparison — val set:")
print(comparison.to_string())

balanced_auc = baseline_metrics["roc_auc"]
smote_auc    = smote_metrics["roc_auc"]
USE_SMOTE    = smote_auc > balanced_auc

winner_name  = "(b) SMOTE + LightGBM" if USE_SMOTE else "(a) class_weight='balanced'"
winner_model = smote_model             if USE_SMOTE else baseline

print(f"\nWinner : {winner_name}")
print(f"  Val ROC-AUC = {max(balanced_auc, smote_auc):.4f}")
if USE_SMOTE:
    print("  SMOTE synthesises new minority-class examples by interpolating between")
    print("  existing churned customers, giving the model richer signal without")
    print("  simply repeating real samples.")
else:
    print("  class_weight='balanced' upweights each churned customer in the loss,")
    print("  achieving strong minority-class recall without altering training data.")

print(f"\nUSE_SMOTE = {USE_SMOTE}  (carries forward into Sections 3 and 4)")

Imbalance approach comparison — val set:
                             accuracy  precision  recall      f1  roc_auc
approach                                                                 
(a) class_weight='balanced'    0.9895      0.973  0.9908  0.9818   0.9999
(b) SMOTE + no class_weight    0.9974      1.000  0.9908  0.9954   0.9999

Winner : (a) class_weight='balanced'
  Val ROC-AUC = 0.9999
  class_weight='balanced' upweights each churned customer in the loss,
  achieving strong minority-class recall without altering training data.

USE_SMOTE = False  (carries forward into Sections 3 and 4)


---
## Section 3 — Hyperparameter Tuning

Manual grid search: **24 combinations** (3 × 2 × 2 × 2).  
Evaluated on the **val set directly** — no cross-validation, which would mix temporal cohorts.

| Parameter | Values |
|---|---|
| `num_leaves` | 31, 63, 127 |
| `learning_rate` | 0.05, 0.1 |
| `n_estimators` | 100, 300 |
| `min_child_samples` | 20, 50 |

In [7]:
PARAM_GRID = {
    "num_leaves":        [31, 63, 127],
    "learning_rate":     [0.05, 0.1],
    "n_estimators":      [100, 300],
    "min_child_samples": [20, 50],
}

grid_results = []
best_auc, best_params, best_grid_model = 0.0, {}, None

keys   = list(PARAM_GRID.keys())
combos = list(product(*PARAM_GRID.values()))
print(f"Running {len(combos)} combinations "
      f"({'SMOTE' if USE_SMOTE else 'class_weight=balanced'})...")

for i, vals in enumerate(combos, 1):
    params = dict(zip(keys, vals))

    if USE_SMOTE:
        mdl = LGBMClassifier(**params, random_state=42, verbose=-1)
        mdl.fit(X_train_sm, y_train_sm)
    else:
        mdl = LGBMClassifier(**params, class_weight="balanced", random_state=42, verbose=-1)
        mdl.fit(X_train, y_train)

    auc = roc_auc_score(y_val, mdl.predict_proba(X_val)[:, 1])
    f1  = f1_score(y_val, mdl.predict(X_val))
    grid_results.append({**params, "val_auc": round(auc, 4), "val_f1": round(f1, 4)})

    if auc > best_auc:
        best_auc, best_params, best_grid_model = auc, params, mdl

grid_df = pd.DataFrame(grid_results).sort_values("val_auc", ascending=False)
print(f"\nTop 5 configs by val ROC-AUC:")
print(grid_df.head(5).to_string(index=False))

Running 24 combinations (class_weight=balanced)...

Top 5 configs by val ROC-AUC:
 num_leaves  learning_rate  n_estimators  min_child_samples  val_auc  val_f1
         31            0.1           300                 50      1.0  0.9954
         31            0.1           100                 50      1.0  0.9954
         63            0.1           100                 50      1.0  0.9954
         63            0.1           300                 50      1.0  0.9954
        127            0.1           300                 50      1.0  0.9954


In [8]:
print(f"Best val ROC-AUC : {best_auc:.4f}")
print(f"Best params:")
for k, v in best_params.items():
    print(f"  {k:<22} = {v}")

Best val ROC-AUC : 1.0000
Best params:
  num_leaves             = 31
  learning_rate          = 0.1
  n_estimators           = 100
  min_child_samples      = 50


---
## Section 4 — Final Model

Refit the best config on **train + val combined** (val data is now "seen" and can be used
for training). The held-out **test set** was never touched until this final evaluation.

In [9]:
train_val = pd.concat([train, val], ignore_index=True)
X_tv = train_val[FEATURE_COLS]
y_tv = train_val[TARGET_COL]

if USE_SMOTE:
    X_tv_fit, y_tv_fit = SMOTE(random_state=42).fit_resample(X_tv, y_tv)
    final_model = LGBMClassifier(**best_params, random_state=42, verbose=-1)
else:
    X_tv_fit, y_tv_fit = X_tv, y_tv
    final_model = LGBMClassifier(
        **best_params, class_weight="balanced", random_state=42, verbose=-1
    )

final_model.fit(X_tv_fit, y_tv_fit)
print(f"Final model trained on {len(X_tv_fit):,} samples "
      f"({'SMOTE-resampled' if USE_SMOTE else 'class_weight=balanced'})")

Final model trained on 2,542 samples (class_weight=balanced)


In [10]:
test_metrics = eval_metrics(final_model, X_test, y_test, split="test")
print("Final model — test set metrics:")
pd.DataFrame([test_metrics]).set_index("split")

Final model — test set metrics:


,accuracy,precision,recall,f1,roc_auc
split,,,,,
test,1.0,1.0,1.0,1.0,1.0


In [11]:
y_prob_test = final_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
auc_val     = roc_auc_score(y_test, y_prob_test)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode="lines",
    name=f"LightGBM (AUC = {auc_val:.3f})",
    line=dict(color="#2196F3", width=2),
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    name="Random baseline",
    line=dict(color="grey", dash="dash"),
))
fig.update_layout(
    title="ROC Curve — test set",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    legend=dict(x=0.6, y=0.1),
)
fig.show()

In [12]:
importance_df = pd.DataFrame({
    "feature":    FEATURE_COLS,
    "importance": final_model.feature_importances_,
}).sort_values("importance", ascending=True)

fig = px.bar(
    importance_df, x="importance", y="feature",
    orientation="h",
    title="LightGBM feature importance (gain)",
    labels={"importance": "Importance", "feature": "Feature"},
    color="importance",
    color_continuous_scale="Blues",
)
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

In [13]:
joblib.dump(final_model, MODEL_PATH)

preds_df = test[["customer_id", TARGET_COL]].copy()
preds_df["churn_prob"] = y_prob_test
preds_df.to_parquet(PREDS_PATH, engine="pyarrow", index=False)

assert MODEL_PATH.exists()
chk = pd.read_parquet(PREDS_PATH)
assert list(chk.columns) == ["customer_id", "churn_label", "churn_prob"]
assert chk["churn_prob"].between(0, 1).all()

print(f"Model saved → {MODEL_PATH.resolve()}")
print(f"Preds saved → {PREDS_PATH.resolve()}  ({len(chk):,} rows)")
print(f"\nTest metrics:")
for k, v in test_metrics.items():
    if k != "split":
        print(f"  {k:<12} {v:.4f}")

Model saved → C:\Users\h11la\OneDrive\Documents\00 Portfolio\customer-analytics-ml-pipeline\retail-clv-churn-prediction\outputs\churn_model.pkl
Preds saved → C:\Users\h11la\OneDrive\Documents\00 Portfolio\customer-analytics-ml-pipeline\retail-clv-churn-prediction\outputs\test_predictions.parquet  (293 rows)

Test metrics:
  accuracy     1.0000
  precision    1.0000
  recall       1.0000
  f1           1.0000
  roc_auc      1.0000
